Create database and load data in the database


In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

Define project paths

In [2]:
PROJECT_DIR = Path.cwd().parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
DATABASE_DIR = PROJECT_DIR / "database"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Database directory:", DATABASE_DIR)

Project directory: c:\Users\sapni\OneDrive\Desktop\Chicago_Crime_Capstone
Raw data directory: c:\Users\sapni\OneDrive\Desktop\Chicago_Crime_Capstone\data\raw
Processed data directory: c:\Users\sapni\OneDrive\Desktop\Chicago_Crime_Capstone\data\processed
Database directory: c:\Users\sapni\OneDrive\Desktop\Chicago_Crime_Capstone\database


Load the cleaned crime dataset
We use the cleaned CSV

In [3]:
cleaned_file = PROCESSED_DATA_DIR / "chicago_crime_dataset_cleaned.csv"

if not cleaned_file.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {cleaned_file}")

crime_df = pd.read_csv(cleaned_file, low_memory=False)

print("Cleaned crime dataset loaded.")
print("Rows:", crime_df.shape[0])
print("Columns:", crime_df.shape[1])

display(crime_df.head())

Cleaned crime dataset loaded.
Rows: 498
Columns: 23


,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,...,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location,Year,Month,DayOfWeek
0,10508693,HZ250496,2016-05-03,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,...,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)",2016,5,Tuesday
1,10508695,HZ250409,2016-05-03,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,...,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)",2016,5,Tuesday
2,10508697,HZ250503,2016-05-03,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,...,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)",2016,5,Tuesday
3,10508698,HZ250424,2016-05-03,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,...,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)",2016,5,Tuesday
4,10508699,HZ250455,2016-05-03,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,...,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)",2016,5,Tuesday


Load the five reference datasets

In [5]:
reference_files = {
    "community": "chicago_city_community.csv",
    "district": "chicago_district_ps_info.csv",
    "beat": "chicago_police_beat_info.csv",
    "ward": "chicago_ward_offices.csv",
    "iucr": "iucr_codes.csv.csv"
}

reference_dfs = {}

for name, filename in reference_files.items():
    file_path = RAW_DATA_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(f"Reference file not found: {file_path}")

    reference_dfs[name] = pd.read_csv(file_path, low_memory=False)

    print(
        f"{name}: {reference_dfs[name].shape[0]} rows, "
        f"{reference_dfs[name].shape[1]} columns"
    )

community: 77 rows, 7 columns
district: 22 rows, 15 columns
beat: 277 rows, 4 columns
ward: 58 rows, 20 columns
iucr: 410 rows, 4 columns


Inspect the reference table columns
We need the exact column names before creating relationships.

In [9]:
for name, df in reference_dfs.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())
    
    
DATABASE_FILE = DATABASE_DIR / "chicago_crime.db"

print("\nDatabase file will be:", DATABASE_FILE)


COMMUNITY COLUMNS
['community_code', 'community_name', 'population', 'area_sqmile', 'area_sqkm', 'density_per_sqmi', 'density_per_sqkm']

DISTRICT COLUMNS
['DISTRICT_CODE', 'DISTRICT_NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'WEBSITE', 'PHONE', 'FAX', 'TTY', 'X_COORDINATE', 'Y_COORDINATE', 'LATITUDE', 'LONGITUDE', 'LOCATION']

BEAT COLUMNS
['DISTRICT', 'SECTOR', 'BEAT', 'BEAT_NUM']

WARD COLUMNS
['ward', 'alderman', 'address', 'city', 'state', 'zipcode', 'ward_phone', 'ward_fax', 'email', 'website', 'location_zip', 'location_city', 'location', 'location_address', 'location_state', 'city_hall_address', 'city_hall_city', 'city_hall_state', 'city_hall_zipcode', 'city_hall_phone']

IUCR COLUMNS
['IUCR', 'PRIMARY_TYPE ', 'DESCRIPTION', 'INDEX CODE']

Database file will be: c:\Users\sapni\OneDrive\Desktop\Chicago_Crime_Capstone\database\chicago_crime.db


Connect to SQLite

In [10]:
connection = sqlite3.connect(DATABASE_FILE)

print("SQLite database connection successful.")

SQLite database connection successful.


Enable foreign-key enforcement
SQLite requires foreign-key enforcement to be enabled for each connection.

In [11]:
connection.execute("PRAGMA foreign_keys = ON")

foreign_keys_status = connection.execute(
    "PRAGMA foreign_keys"
).fetchone()[0]

print("Foreign keys enabled:", foreign_keys_status == 1)

Foreign keys enabled: True


Inspect the keys before creating tables
We should not create foreign-key relationships until we verify that the reference tables contain valid, unique keys.

In [12]:
key_columns = {
    "crime": ["case_number", "iucr_code", "community_code",
              "district_code", "beat_num", "ward_no"],
    "community": ["community_code"],
    "district": ["DISTRICT_CODE"],
    "beat": ["BEAT_NUM"],
    "ward": ["ward"],
    "iucr": ["IUCR"]
}

all_dfs = {
    "crime": crime_df,
    **reference_dfs
}

for table_name, columns in key_columns.items():
    df = all_dfs[table_name]

    print(f"\n{table_name.upper()}")

    for column in columns:
        if column in df.columns:
            print(
                f"{column}: "
                f"{df[column].notna().sum()} non-missing values, "
                f"{df[column].nunique()} unique values"
            )
        else:
            print(f"{column}: COLUMN NOT FOUND")


CRIME
case_number: 498 non-missing values, 498 unique values
iucr_code: 498 non-missing values, 97 unique values
community_code: 498 non-missing values, 70 unique values
district_code: 498 non-missing values, 22 unique values
beat_num: 498 non-missing values, 209 unique values
ward_no: 498 non-missing values, 50 unique values

COMMUNITY
community_code: 77 non-missing values, 77 unique values

DISTRICT
DISTRICT_CODE: 22 non-missing values, 22 unique values

BEAT
BEAT_NUM: 277 non-missing values, 275 unique values

WARD
ward: 58 non-missing values, 50 unique values

IUCR
IUCR: 410 non-missing values, 410 unique values


Check duplicate reference keys
A reference key should normally identify one row. This cell checks whether the proposed keys are unique.

In [13]:
reference_key_columns = {
    "community": "community_code",
    "district": "DISTRICT_CODE",
    "beat": "BEAT_NUM",
    "ward": "ward",
    "iucr": "IUCR"
}

for table_name, key_column in reference_key_columns.items():
    df = reference_dfs[table_name]

    print(f"\n{table_name.upper()} — {key_column}")

    if key_column not in df.columns:
        print("Column not found.")
        continue

    print("Missing keys:", df[key_column].isna().sum())
    print("Duplicate keys:", df[key_column].duplicated().sum())
    print("Unique keys:", df[key_column].nunique())


COMMUNITY — community_code
Missing keys: 0
Duplicate keys: 0
Unique keys: 77

DISTRICT — DISTRICT_CODE
Missing keys: 0
Duplicate keys: 0
Unique keys: 22

BEAT — BEAT_NUM
Missing keys: 0
Duplicate keys: 2
Unique keys: 275

WARD — ward
Missing keys: 0
Duplicate keys: 8
Unique keys: 50

IUCR — IUCR
Missing keys: 0
Duplicate keys: 0
Unique keys: 410


Inspect sample key values
This helps us confirm that the master dataset's codes match the reference datasets.

In [14]:
sample_keys = {
    "crime": ["iucr_code", "community_code", "district_code",
              "beat_num", "ward_no"],
    "community": ["community_code"],
    "district": ["DISTRICT_CODE"],
    "beat": ["BEAT_NUM"],
    "ward": ["ward"],
    "iucr": ["IUCR"]
}

for table_name, columns in sample_keys.items():
    df = all_dfs[table_name]

    print(f"\n{table_name.upper()} KEY SAMPLES")

    for column in columns:
        if column in df.columns:
            print(f"\n{column}")
            print(df[column].dropna().astype(str).head(10).tolist())


CRIME KEY SAMPLES

iucr_code
['486', '486', '470', '460', '820', '041A', '460', '460', '460', '486']

community_code
['29', '42', '25', '25', '25', '44', '35', '38', '1', '67']

district_code
['10', '3', '15', '15', '15', '6', '1', '2', '24', '7']

beat_num
['1022', '313', '1524', '1532', '1523', '631', '133', '215', '2432', '735']

ward_no
['24', '20', '37', '28', '28', '8', '3', '3', '40', '17']

COMMUNITY KEY SAMPLES

community_code
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']

DISTRICT KEY SAMPLES

DISTRICT_CODE
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']

BEAT KEY SAMPLES

BEAT_NUM
['1713', '3100', '1651', '1914', '1915', '1913', '735', '1912', '1723', '1911']

WARD KEY SAMPLES

ward
['42', '34', '46', '3', '38', '37', '6', '17', '5', '15']

IUCR KEY SAMPLES

IUCR
['110', '130', '141', '142', '261', '262', '263', '264', '265', '266']


Display duplicate beat keys

In [18]:
beat_df = reference_dfs["beat"].copy()

duplicate_beats = beat_df[
    beat_df["BEAT_NUM"].duplicated(keep=False)
].sort_values("BEAT_NUM")

print("Duplicate beat records:", len(duplicate_beats))

display(duplicate_beats)

for beat_number, group in duplicate_beats.groupby("BEAT_NUM"):
    print(f"\nBeat number: {beat_number}")
    display(group)
    

Duplicate beat records: 3


,DISTRICT,SECTOR,BEAT,BEAT_NUM
1,31,0,0,3100
55,31,0,0,3100
263,31,0,0,3100



Beat number: 3100


,DISTRICT,SECTOR,BEAT,BEAT_NUM
1,31,0,0,3100
55,31,0,0,3100
263,31,0,0,3100


Display duplicate ward keys

In [19]:
ward_df = reference_dfs["ward"].copy()

duplicate_wards = ward_df[
    ward_df["ward"].duplicated(keep=False)
].sort_values("ward")

print("Duplicate ward records:", len(duplicate_wards))

display(duplicate_wards)

Duplicate ward records: 16


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
36,2,"Hopkins, Brian",1400 North Ashland Avenue,Chicago,IL,60622.0,(312) 643-2299,(773) 270-4688,Ward02@cityofchicago.org,https://www.aldermanhopkins.com/,NaN,NaN,POINT (-87.667668 41.907081),NaN,NaN,"121 North LaSalle Street, Room 200, Office 8",Chicago,IL,60602.0,(312) 744-6836
51,2,"Hopkins, Brian","121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6836 / 9867/6213,NaN,office@aldermanhopkins.com,https://www.aldermanhopkins.com/,NaN,NaN,POINT (-87.63227535353653 41.8837645981034),NaN,NaN,NaN,NaN,NaN,NaN,NaN
50,6,"Hall, William E.",8541 South State Street,Chicago,IL,60619.0,(773) 241-3100,NaN,Ward06@cityofchicago.org,NaN,NaN,NaN,POINT (-87.62405576399621 41.73878805068228),NaN,NaN,121 N. LaSalle St. Room 300,Chicago,IL,60602.0,(312) 744-6868
6,6,"Sawyer, Roderick T.",700 East 79th Street,Chicago,IL,60619.0,(773) 635-0006,(773) 891-5679,Ward06@cityofchicago.org,http://www.6ward.com,NaN,NaN,POINT (-87.607315 41.75124),NaN,NaN,"121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6868
23,20,"Taylor, Jeanette B.",5707 South Wentworth Avenue,Chicago,IL,60621.0,(773) 966-5336,NaN,Jeanette.Taylor@cityofchicago.org,NaN,NaN,NaN,POINT (-87.630388 41.7905),NaN,NaN,"121 North LaSalle Street, Room 300, Office 40",Chicago,IL,60602.0,(312) 744-6840
55,20,"Taylor, Jeanette B.","5401 South Wentworth Avenue, Suite 19E",Chicago,IL,60621.0,(773) 966-5336,NaN,Ward20@cityofchicago.org,NaN,NaN,NaN,POINT (-87.63040492438661 41.796234875219376),NaN,NaN,"121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6840/5688/7738
57,27,"Burnett, Walter R.","4 North Western Avenue, Unit 1C",Chicago,IL,60618.0,(312) 432-1995,(312) 432-1049,Ward27@cityofchicago.org,http://aldermanburnett.com/,NaN,NaN,POINT (-87.68658173977015 41.88126893284509),NaN,NaN,"121 North LaSalle Street, Room 300, Office 38",Chicago,IL,60602.0,(312) 744-6124/6125
47,27,"Burnett, Jr., Walter",4 North Western Avenue,Chicago,IL,60612.0,(312) 432-1995,(312) 432-1049,Ward27@cityofchicago.org,http://aldermanburnett.com/,NaN,NaN,POINT (-87.686515 41.881257),NaN,NaN,"121 North LaSalle Street, Room 300, Office 38",Chicago,IL,60602.0,(312) 744-6124 / 6125
52,30,"Cruz, Ruth",5418 W. Belmont Ave,Chicago,IL,60641.0,(773) 628-7874,NaN,Ward30@cityofchicago.org,NaN,NaN,NaN,POINT (-87.76236826862518 41.93873912421377),NaN,NaN,121 N. LaSalle St. Room 200,Chicago,IL,60602.0,(312) 744-3304/3305
21,30,"Reboyras, Ariel",3559 North Milwaukee Avenue,Chicago,IL,60641.0,(773) 794-3095,(773) 794-8576,Ward30@cityofchicago.org,https://www.30thwardchicago.com/,NaN,NaN,POINT (-87.733966 41.945277),NaN,NaN,"121 North LaSalle Street, Room 200, Office 17",Chicago,IL,60602.0,(312) 744-3304 / 3305


Inspect duplicate ward details

In [20]:
for ward_number, group in duplicate_wards.groupby("ward"):
    print(f"\nWard number: {ward_number}")
    display(group)


Ward number: 2


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
36,2,"Hopkins, Brian",1400 North Ashland Avenue,Chicago,IL,60622.0,(312) 643-2299,(773) 270-4688,Ward02@cityofchicago.org,https://www.aldermanhopkins.com/,NaN,NaN,POINT (-87.667668 41.907081),NaN,NaN,"121 North LaSalle Street, Room 200, Office 8",Chicago,IL,60602.0,(312) 744-6836
51,2,"Hopkins, Brian","121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6836 / 9867/6213,NaN,office@aldermanhopkins.com,https://www.aldermanhopkins.com/,NaN,NaN,POINT (-87.63227535353653 41.8837645981034),NaN,NaN,NaN,NaN,NaN,NaN,NaN



Ward number: 6


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
50,6,"Hall, William E.",8541 South State Street,Chicago,IL,60619.0,(773) 241-3100,NaN,Ward06@cityofchicago.org,NaN,NaN,NaN,POINT (-87.62405576399621 41.73878805068228),NaN,NaN,121 N. LaSalle St. Room 300,Chicago,IL,60602.0,(312) 744-6868
6,6,"Sawyer, Roderick T.",700 East 79th Street,Chicago,IL,60619.0,(773) 635-0006,(773) 891-5679,Ward06@cityofchicago.org,http://www.6ward.com,NaN,NaN,POINT (-87.607315 41.75124),NaN,NaN,"121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6868



Ward number: 20


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
23,20,"Taylor, Jeanette B.",5707 South Wentworth Avenue,Chicago,IL,60621.0,(773) 966-5336,NaN,Jeanette.Taylor@cityofchicago.org,NaN,NaN,NaN,POINT (-87.630388 41.7905),NaN,NaN,"121 North LaSalle Street, Room 300, Office 40",Chicago,IL,60602.0,(312) 744-6840
55,20,"Taylor, Jeanette B.","5401 South Wentworth Avenue, Suite 19E",Chicago,IL,60621.0,(773) 966-5336,NaN,Ward20@cityofchicago.org,NaN,NaN,NaN,POINT (-87.63040492438661 41.796234875219376),NaN,NaN,"121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6840/5688/7738


... (6 outputs left out)


Ward number: 35


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
53,35,"Quezada, Anthony J.","2934 North Milwaukee Avenue, Unit C",Chicago,IL,60618.0,(773) 887-3772,773) 887-5739,ward35@cityofchicago.org,NaN,NaN,NaN,POINT (-87.71650804469363 41.93447944793928),NaN,NaN,"121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6835/3109/6845
37,35,"Ramirez-Rosa, Carlos","2934 North Milwaukee Avenue, Unit C",Chicago,IL,60618.0,(773) 887-3772,(773) 887-5739,Ward35@cityofchicago.org,http://www.aldermancarlosrosa.org,NaN,NaN,POINT (-87.716556 41.934652),NaN,NaN,"121 North LaSalle Street, Room 200, Office 9",Chicago,IL,60602.0,(312) 744-6835



Ward number: 45


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
28,45,"Gardiner, James M.",5425 West Lawrence Avenue,Chicago,IL,60630.0,(773) 853-0799,NaN,ward45@cityofchicago.org,http://www.aldermangardiner.com,NaN,NaN,POINT (-87.763536 41.96774),NaN,NaN,"121 North LaSalle Street, Room 300, Office 44",Chicago,IL,60602.0,(312) 744-6841
54,45,"Gardiner, James M.",5460 N. Milwaukee Ave,Chicago,IL,60630.0,(773) 853-0799,NaN,Ward45@cityofchicago.org,http://www.aldermangardiner.com,NaN,NaN,POINT (-87.77253948587003 41.980733083375114),NaN,NaN,"121 North LaSalle Street, Room 200",Chicago,IL,60602.0,(312) 744-6841/5684


Create standardized copies

In [21]:
crime_keys_df = crime_df.copy()

community_df = reference_dfs["community"].copy()
district_df = reference_dfs["district"].copy()
beat_df = reference_dfs["beat"].copy()
ward_df = reference_dfs["ward"].copy()
iucr_df = reference_dfs["iucr"].copy()

# Remove spaces from column names
for df in [
    crime_keys_df,
    community_df,
    district_df,
    beat_df,
    ward_df,
    iucr_df
]:
    df.columns = df.columns.str.strip()

# Standardize key values as text
key_standardization = [
    (crime_keys_df, ["case_number", "iucr_code", "community_code",
                     "district_code", "beat_num", "ward_no"]),
    (community_df, ["community_code"]),
    (district_df, ["DISTRICT_CODE"]),
    (beat_df, ["BEAT_NUM"]),
    (ward_df, ["ward"]),
    (iucr_df, ["IUCR"])
]

for df, columns in key_standardization:
    for column in columns:
        df[column] = df[column].astype("string").str.strip()

print("Key columns standardized.")

Key columns standardized.


Check unmatched community codes

In [22]:
crime_community_codes = set(
    crime_keys_df["community_code"].dropna()
)

reference_community_codes = set(
    community_df["community_code"].dropna()
)

unmatched_community = crime_community_codes - reference_community_codes

print("Unmatched community codes:", unmatched_community)

Unmatched community codes: set()


Check unmatched district codes

In [23]:
crime_district_codes = set(
    crime_keys_df["district_code"].dropna()
)

reference_district_codes = set(
    district_df["DISTRICT_CODE"].dropna()
)

unmatched_district = crime_district_codes - reference_district_codes

print("Unmatched district codes:", unmatched_district)

Unmatched district codes: set()


Check unmatched beat numbers

In [24]:
crime_beat_numbers = set(
    crime_keys_df["beat_num"].dropna()
)

reference_beat_numbers = set(
    beat_df["BEAT_NUM"].dropna()
)

unmatched_beats = crime_beat_numbers - reference_beat_numbers

print("Unmatched beat numbers:", unmatched_beats)

Unmatched beat numbers: set()


Check unmatched ward numbers

In [25]:
crime_ward_numbers = set(
    crime_keys_df["ward_no"].dropna()
)

reference_ward_numbers = set(
    ward_df["ward"].dropna()
)

unmatched_wards = crime_ward_numbers - reference_ward_numbers

print("Unmatched ward numbers:", unmatched_wards)

Unmatched ward numbers: set()


Check unmatched IUCR codes

In [26]:
crime_iucr_codes = set(
    crime_keys_df["iucr_code"].dropna()
)

reference_iucr_codes = set(
    iucr_df["IUCR"].dropna()
)

unmatched_iucr = crime_iucr_codes - reference_iucr_codes

print("Unmatched IUCR codes:", unmatched_iucr)

Unmatched IUCR codes: set()


Check duplicate case numbers
The crime table's case_number will be its proposed primary key.

Verify the crime primary key

In [27]:
print("Total crime records:", len(crime_keys_df))
print("Missing case numbers:", crime_keys_df["case_number"].isna().sum())
print("Unique case numbers:", crime_keys_df["case_number"].nunique())
print("Duplicate case numbers:", crime_keys_df["case_number"].duplicated().sum())

Total crime records: 498
Missing case numbers: 0
Unique case numbers: 498
Duplicate case numbers: 0


Run final key validation

In [28]:
print("REFERENCE KEY VALIDATION")

print("Unmatched IUCR codes:", unmatched_iucr)
print("Unmatched community codes:", unmatched_community)
print("Unmatched district codes:", unmatched_district)
print("Unmatched beat numbers:", unmatched_beats)
print("Unmatched ward numbers:", unmatched_wards)

print("\nCRIME PRIMARY KEY VALIDATION")

print("Total crime records:", len(crime_keys_df))
print("Missing case numbers:", crime_keys_df["case_number"].isna().sum())
print("Unique case numbers:", crime_keys_df["case_number"].nunique())
print("Duplicate case numbers:", crime_keys_df["case_number"].duplicated().sum())

REFERENCE KEY VALIDATION
Unmatched IUCR codes: set()
Unmatched community codes: set()
Unmatched district codes: set()
Unmatched beat numbers: set()
Unmatched ward numbers: set()

CRIME PRIMARY KEY VALIDATION
Total crime records: 498
Missing case numbers: 0
Unique case numbers: 498
Duplicate case numbers: 0


Clean the beat reference table

In [29]:
beat_df_cleaned = beat_df.drop_duplicates().copy()

print("Original beat records:", len(beat_df))
print("Cleaned beat records:", len(beat_df_cleaned))
print("Duplicate beat numbers remaining:",
      beat_df_cleaned["BEAT_NUM"].duplicated().sum())

Original beat records: 277
Cleaned beat records: 275
Duplicate beat numbers remaining: 0


Prepare the ward reference table
We retain all records and create a new unique ID.

In [30]:
ward_df_cleaned = ward_df.copy().reset_index(drop=True)

ward_df_cleaned.insert(
    0,
    "ward_office_id",
    range(1, len(ward_df_cleaned) + 1)
)

print("Ward office records retained:", len(ward_df_cleaned))
print("Unique office IDs:", ward_df_cleaned["ward_office_id"].nunique())
print("Unique ward numbers:", ward_df_cleaned["ward"].nunique())

Ward office records retained: 58
Unique office IDs: 58
Unique ward numbers: 50


Check year consistency

In [31]:
year_check = pd.to_numeric(
    crime_df["year"],
    errors="coerce"
)

date_year = pd.to_datetime(
    crime_df["date"],
    errors="coerce"
).dt.year

year_mismatches = crime_df.loc[
    year_check != date_year,
    ["case_number", "date", "year"]
]

print("Year mismatches:", len(year_mismatches))
display(year_mismatches.head(10))

Year mismatches: 0


,case_number,date,year


Prepare all six dataframes

In [37]:
# Create clean copies for database insertion

crime_db = crime_df.copy()

community_db = community_df.copy()
district_db = district_df.copy()
beat_db = beat_df_cleaned.copy()
ward_db = ward_df_cleaned.copy()
iucr_db = iucr_df.copy()

# Remove accidental spaces from column names
for df in [
    crime_db,
    community_db,
    district_db,
    beat_db,
    ward_db,
    iucr_db
]:
    df.columns = df.columns.str.strip()

# Remove derived Year column from database copy only
# Keep the original year column
if "Year" in crime_db.columns:
    crime_db = crime_db.drop(columns=["Year"])

print("Columns prepared for database:")
print(crime_db.columns.tolist())

print("\nOriginal year column retained:", "year" in crime_db.columns)
print("Derived Year column removed:", "Year" not in crime_db.columns)

Columns prepared for database:
['id', 'case_number', 'date', 'block', 'iucr_code', 'location_desc', 'arrest', 'domestic', 'beat_num', 'district_code', 'ward_no', 'community_code', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'date_of_update', 'latitude', 'longitude', 'location', 'Month', 'DayOfWeek']

Original year column retained: True
Derived Year column removed: True


Check dataframe dimensions

In [38]:
for table_name, df in database_dfs.items():
    print(
        f"{table_name}: {df.shape[0]} rows, "
        f"{df.shape[1]} columns"
    )

chicago_crime: 498 rows, 23 columns
city_community: 77 rows, 7 columns
district_ps_info: 22 rows, 15 columns
police_beat_info: 275 rows, 4 columns
ward_office: 58 rows, 21 columns
iucr: 410 rows, 4 columns


Create the SQLite database schema

Define a helper to generate SQL column definitions

In [39]:
def get_sql_type(series):
    if pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    elif pd.api.types.is_float_dtype(series):
        return "REAL"
    else:
        return "TEXT"


def quote_identifier(column_name):
    return '"' + column_name.replace('"', '""') + '"'

Create the reference tables

In [41]:
reference_table_configs = {
    "iucr": {
        "dataframe": iucr_db,
        "primary_key": "IUCR"
    },
    "city_community": {
        "dataframe": community_db,
        "primary_key": "community_code"
    },
    "district_ps_info": {
        "dataframe": district_db,
        "primary_key": "DISTRICT_CODE"
    },
    "police_beat_info": {
        "dataframe": beat_db,
        "primary_key": "BEAT_NUM"
    },
    "ward_office": {
        "dataframe": ward_db,
        "primary_key": "ward_office_id"
    }
}

for table_name, config in reference_table_configs.items():

    df = config["dataframe"]
    primary_key = config["primary_key"]

    column_definitions = []

    for column in df.columns:
        column_sql_type = get_sql_type(df[column])

        if column == primary_key:
            if table_name == "ward_office":
                definition = f'{quote_identifier(column)} INTEGER PRIMARY KEY'
            else:
                definition = (
                    f'{quote_identifier(column)} TEXT PRIMARY KEY'
                )
        else:
            definition = (
                f'{quote_identifier(column)} {column_sql_type}'
            )

        column_definitions.append(definition)

    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {quote_identifier(table_name)} (
        {", ".join(column_definitions)}
    );
    """

    connection.execute(create_sql)

connection.commit()

print("Reference tables created successfully.")

Reference tables created successfully.


Create the main crime table with foreign keys

In [42]:
crime_column_definitions = []

for column in crime_db.columns:

    column_sql_type = get_sql_type(crime_db[column])

    if column == "case_number":
        definition = '"case_number" TEXT PRIMARY KEY'

    elif column in [
        "iucr_code",
        "community_code",
        "district_code",
        "beat_num"
    ]:
        definition = f'"{column}" TEXT'

    else:
        definition = f'{quote_identifier(column)} {column_sql_type}'

    crime_column_definitions.append(definition)

foreign_key_definitions = [
    'FOREIGN KEY ("iucr_code") REFERENCES "iucr" ("IUCR")',
    'FOREIGN KEY ("community_code") REFERENCES "city_community" ("community_code")',
    'FOREIGN KEY ("district_code") REFERENCES "district_ps_info" ("DISTRICT_CODE")',
    'FOREIGN KEY ("beat_num") REFERENCES "police_beat_info" ("BEAT_NUM")'
]

create_crime_sql = f"""
CREATE TABLE IF NOT EXISTS chicago_crime (
    {", ".join(crime_column_definitions)},
    {", ".join(foreign_key_definitions)}
);
"""

connection.execute(create_crime_sql)
connection.commit()

print("Main crime table created successfully.")

Main crime table created successfully.


Insert reference datasets
It inserts the five reference datasets.

In [43]:
reference_table_configs = {
    "iucr": {
        "dataframe": iucr_db,
        "primary_key": "IUCR"
    },
    "city_community": {
        "dataframe": community_db,
        "primary_key": "community_code"
    },
    "district_ps_info": {
        "dataframe": district_db,
        "primary_key": "DISTRICT_CODE"
    },
    "police_beat_info": {
        "dataframe": beat_db,
        "primary_key": "BEAT_NUM"
    },
    "ward_office": {
        "dataframe": ward_db,
        "primary_key": "ward_office_id"
    }
}

for table_name, config in reference_table_configs.items():
    df = config["dataframe"]

    df.to_sql(
        name=table_name,
        con=connection,
        if_exists="append",
        index=False
    )

    print(f"{table_name}: {len(df)} records inserted")

iucr: 410 records inserted
city_community: 77 records inserted
district_ps_info: 22 records inserted
police_beat_info: 275 records inserted
ward_office: 58 records inserted


Insert crime records

In [44]:
crime_db.to_sql(
    name="chicago_crime",
    con=connection,
    if_exists="append",
    index=False
)

print("Crime records inserted:", len(crime_db))

Crime records inserted: 498


Commit the changes


In [ ]:
connection.commit()

print("All database insertions committed successfully.")

Verify database row counts

In [45]:
table_names = [
    "chicago_crime",
    "iucr",
    "city_community",
    "district_ps_info",
    "police_beat_info",
    "ward_office"
]

for table_name in table_names:
    count = connection.execute(
        f'SELECT COUNT(*) FROM "{table_name}"'
    ).fetchone()[0]

    print(f"{table_name}: {count} records")

chicago_crime: 498 records
iucr: 410 records
city_community: 77 records
district_ps_info: 22 records
police_beat_info: 275 records
ward_office: 58 records


Verify foreign-key integrity

In [46]:
foreign_key_errors = connection.execute(
    "PRAGMA foreign_key_check"
).fetchall()

print("Foreign key errors:", foreign_key_errors)

if len(foreign_key_errors) == 0:
    print("Foreign key validation PASSED.")
else:
    print("Foreign key validation FAILED.")

Foreign key errors: []
Foreign key validation PASSED.


Verify primary keys

In [47]:
for table_name in table_names:
    result = connection.execute(
        f'PRAGMA table_info("{table_name}")'
    ).fetchall()

    primary_keys = [
        row[1] for row in result if row[5] == 1
    ]

    print(f"{table_name} primary key:", primary_keys)

chicago_crime primary key: ['case_number']
iucr primary key: ['IUCR']
city_community primary key: ['community_code']
district_ps_info primary key: ['DISTRICT_CODE']
police_beat_info primary key: ['BEAT_NUM']
ward_office primary key: ['ward_office_id']


Verify missing geographic coordinates

In [48]:
coordinate_check = connection.execute("""
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) AS missing_latitude,
    SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END) AS missing_longitude,
    SUM(CASE WHEN x_coordinate IS NULL THEN 1 ELSE 0 END) AS missing_x_coordinate,
    SUM(CASE WHEN y_coordinate IS NULL THEN 1 ELSE 0 END) AS missing_y_coordinate
FROM chicago_crime;
""").fetchone()

print("Total records:", coordinate_check[0])
print("Missing latitude:", coordinate_check[1])
print("Missing longitude:", coordinate_check[2])
print("Missing X coordinate:", coordinate_check[3])
print("Missing Y coordinate:", coordinate_check[4])

Total records: 498
Missing latitude: 29
Missing longitude: 29
Missing X coordinate: 29
Missing Y coordinate: 29


Test a SQL join
This tests whether the crime records can successfully join to the IUCR reference table.

In [49]:
sample_query = """
SELECT
    c.case_number,
    c.date,
    c.iucr_code,
    i.PRIMARY_TYPE AS primary_type,
    i.DESCRIPTION AS crime_description,
    c.arrest,
    c.community_code
FROM chicago_crime AS c
LEFT JOIN iucr AS i
    ON c.iucr_code = i.IUCR
LIMIT 10;
"""

sample_result = pd.read_sql_query(
    sample_query,
    connection
)

display(sample_result)

,case_number,date,iucr_code,primary_type,crime_description,arrest,community_code
0,HZ250496,2016-05-03,486,BATTERY,DOMESTIC BATTERY SIMPLE,1,29
1,HZ250409,2016-05-03,486,BATTERY,DOMESTIC BATTERY SIMPLE,0,42
2,HZ250503,2016-05-03,470,PUBLIC PEACE VIOLATION,RECKLESS CONDUCT,0,25
3,HZ250424,2016-05-03,460,BATTERY,SIMPLE,0,25
4,HZ250455,2016-05-03,820,THEFT,$500 AND UNDER,0,25
5,HZ250447,2016-05-03,041A,BATTERY,AGGRAVATED - HANDGUN,0,44
6,HZ250489,2016-05-03,460,BATTERY,SIMPLE,0,35
7,HZ250514,2016-05-03,460,BATTERY,SIMPLE,0,38
8,HZ250523,2016-05-03,460,BATTERY,SIMPLE,0,1
9,HZ250667,2016-05-03,486,BATTERY,DOMESTIC BATTERY SIMPLE,0,67


Close the database connection

In [50]:
connection.close()

print("SQLite database connection closed.")

SQLite database connection closed.
